In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup


In [ ]:
nutrients = {

    "Vitamin C": "VitaminC",
    "Vitamin A": "VitaminA",
    "Vitamin D": "VitaminD",
    "Vitamin E": "VitaminE",
    "Vitamin B6": "VitaminB6",    "Vitamin B12": "VitaminB12",
    "Calcium": "Calcium",
    "Iron": "Iron",
    "Zinc": "Zinc",
   "Magnesium": "Magnesium",

}
base_url = "https://ods.od.nih.gov/factsheets/{}-HealthProfessional/"
columns = ["disease", "nutrient", "min-value", "max-value", "unit", "age-group", "gender"]

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; AcademicNutritionDataset/1.0)"})


In [ ]:
def parse_number(value):
    text = str(value).replace(",", "").strip()
    if text.lower() in {"", "nan", "nd", "n/a", "not established", "none", "—", "-"}:
        return None
    match = re.search(r"(?<!\w)\d+(?:\.\d+)?", text)
    return float(match.group()) if match else None

def find_table(soup, keywords):
    for table in soup.find_all("table"):
        caption = table.find("caption")
        caption_text = caption.get_text(" ", strip=True) if caption else ""

        heading = table.find_previous(["h2", "h3", "h4"])
        heading_text = heading.get_text(" ", strip=True) if heading else ""

        text = (caption_text + " " + heading_text).lower()
        if any(keyword.lower() in text for keyword in keywords):
            return table
    return None

In [ ]:
def adult_rows(table):
    if table is None:
        return []

    rows = []
    for tr in table.find_all("tr"):
        cells = [cell.get_text(" ", strip=True) for cell in tr.find_all(["th", "td"], recursive=False)]

        if len(cells) < 3:
            continue
        age = cells[0].lower().replace("–", "-").replace("—", "-").replace("−", "-")
        if re.search(r"\b19\s*[-–]\s*30\b", age):
            rows.append(cells)
    return rows

def table_unit(table):
    if table is None:
        return None
    header = " ".join(th.get_text(" ", strip=True) for th in table.find_all("th", limit=12))
    caption = table.find("caption")
    label = (caption.get_text(" ", strip=True) if caption else "") + " " + header
    match = re.search(r"\b(mcg|µg|μg|mg|g|IU)\b", label, re.I)
    return match.group(1).replace("μg", "mcg").replace("µg", "mcg") if match else None

def extract_values(table):
    rows = adult_rows(table)
    if not rows:

        return {}
    cells = rows[0]
    return {"Male": parse_number(cells[1]), "Female": parse_number(cells[2])}


In [ ]:
records = []
errors = []
for nutrient, slug in nutrients.items():
    url = base_url.format(slug)
    try:
        response = session.get(url, timeout=25)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        rda_table = find_table(soup, ["Recommended Dietary Allowances", "Adequate Intakes"])
        ul_table = find_table(soup, ["Tolerable Upper Intake Levels", "Tolerable Upper Intake Level"])
        minimums = extract_values(rda_table)
        maximums = extract_values(ul_table)
        unit = table_unit(rda_table) or table_unit(ul_table)

        if not minimums and not maximums:
            errors.append((nutrient, "Adult 19-30 rows not found"))
            continue



        for gender in ("Male", "Female"):
            records.append({
                "disease": "Healthy",
                "nutrient": nutrient,
                "min-value": minimums.get(gender),
                "max-value": maximums.get(gender),
                "unit": unit,
                "age-group": "Adult",
                "gender": gender,
            })
        print("Processed:", nutrient)
    except requests.RequestException as exc:
        errors.append((nutrient, str(exc)))
    time.sleep(0.5)

In [ ]:
df = pd.DataFrame(records, columns=columns)
df.to_csv("nih_nutrient_dataset.csv", index=False)
print("Rows saved:", len(df))
print("Nutrients needing review:", errors)
df